## Classification Results

The enrichment flags parcels as truck/logistics-likely based on:

### ✅ True Positives (Flagged)
- **Strong Keywords**: "TRUCK STOP", "LOGISTICS", "DISTRIBUTION", "WAREHOUSE", chain names (Pilot, Love's, Iowa 80, FedEx, etc.)
- **Industrial Zoning**: `zoning_type = "Industrial"` or `"Manufacturing"`
- **LBCS Codes**: Activity codes in 3000–4999 range (manufacturing, warehousing, transportation)
- **Special Commercial + Weak Keywords**: `zoning_subtype = "Special Commercial"` + "travel", "fuel", "plaza"

### ❌ False Positives (May be flagged but not actually trucking)
- Small commercial plazas with gas stations (special commercial + weak keywords)
- Vacant industrial lots (industrial zoning but not currently in use)
- Public roads/ROW near industrial areas (DOT parcels with industrial signals)

### 📊 Confidence Levels
- **High**: Strong keywords + industrial zoning/LBCS, OR industrial zoning AND LBCS code match
- **Medium**: Strong keywords alone, OR zoning/LBCS without keywords, OR industrial zoning + LBCS without keywords
- **Low**: Special commercial + weak keyword match (many false positives)

## Next Steps

1. **Validate mock results** — Review flagged records in output CSV
2. **Set Regrid API URL**:
   ```bash
   export REGRID_QUERY_URL="https://fs.regrid.com/<your-token>/rest/services/premium/FeatureServer/0/query"
   ```
3. **Update cell 2**: Change `USE_MOCK = False` to hit real Regrid API
4. **Adjust rate limiting**: Set `MIN_INTERVAL_SECS` based on your Regrid plan
5. **Load real prospects**: Replace sample data in cell 6 with your actual Glue Catalog table
6. **Run for production**: Set `LIMIT_ROWS = None` to process all rows
7. **Monitor for errors**: Watch for 429 rate-limit errors; increase `MIN_INTERVAL_SECS` if needed

In [ ]:
# --- Write Results to S3 (or local CSV for testing) ---

# Write to local CSV
result_df.to_csv(OUTPUT_LOCAL_CSV, index=False)
print(f"✓ Wrote {len(result_df)} rows to {OUTPUT_LOCAL_CSV}")

# Display sample results
print("\n📊 Sample Output (first 3 rows):")
print(result_df[["point_id", "lat", "lon", "truck_flag", "confidence", "parcel_owner", "county"]].head(3).to_string())

# Option: Write to S3 using awswrangler (if running in Glue)
# try:
#     if wr:
#         wr.s3.to_parquet(df=result_df, path=OUTPUT_S3_PATH, dataset=True)
#         print(f"✓ Wrote to S3: {OUTPUT_S3_PATH}")
# except Exception as e:
#     print(f"⚠️  S3 write failed: {e}")

In [ ]:
# --- Enrich with Regrid Data ---

enriched_rows = []

print(f"\n[enrichment] Processing {len(df)} prospects...\n")

for idx, row in df.iterrows():
    point_id = str(row.get("point_id", f"P{idx}"))
    lat = float(row["lat"])
    lon = float(row["lon"])
    
    # Query Regrid
    parcel_attrs = None
    error = None
    
    if USE_MOCK:
        # Mock: Return synthetic data
        parcel_attrs = {
            "owner": "SAMPLE WAREHOUSE LLC" if idx % 2 == 0 else "JOHN Q RESIDENTIAL",
            "address": "100 INDUSTRIAL DR" if idx % 2 == 0 else "1 MAIN ST",
            "zoning_type": "Industrial" if idx % 2 == 0 else "Residential",
            "zoning_subtype": "Industrial" if idx % 2 == 0 else "Single Family",
            "lbcs_activity": "3900" if idx % 2 == 0 else "1100",
            "county": row.get("state", "XX").lower(),
            "state2": row.get("state", "XX"),
        }
    else:
        try:
            parcel_attrs = client.query_point(lat, lon)
        except Exception as exc:
            error = str(exc)
            print(f"  Error at {point_id}: {error}")
    
    # Classify
    if parcel_attrs or error:
        classification = classify_parcel(parcel_attrs) if parcel_attrs else {"truck_flag": False, "confidence": "error", "reasons": [error or "unknown"]}
    else:
        classification = {"truck_flag": False, "confidence": "high", "reasons": ["no parcel found"]}
    
    # Build output row
    out_row = {
        "point_id": point_id,
        "lat": lat,
        "lon": lon,
        "truck_flag": classification["truck_flag"],
        "confidence": classification["confidence"],
        "reasons": ";".join(classification["reasons"]),
        "parcel_owner": parcel_attrs.get("owner", "") if parcel_attrs else "",
        "parcel_address": parcel_attrs.get("address", "") if parcel_attrs else "",
        "parcel_zoning_type": parcel_attrs.get("zoning_type", "") if parcel_attrs else "",
        "parcel_zoning_subtype": parcel_attrs.get("zoning_subtype", "") if parcel_attrs else "",
        "county": parcel_attrs.get("county", "") if parcel_attrs else "",
        "state": parcel_attrs.get("state2", "") if parcel_attrs else "",
        "error": error or "",
    }
    enriched_rows.append(out_row)
    
    if (idx + 1) % 5 == 0 or idx == len(df) - 1:
        flagged = sum(1 for r in enriched_rows if r["truck_flag"])
        print(f"  {idx + 1}/{len(df)} done | flagged={flagged}")

# Convert to DataFrame
result_df = pd.DataFrame(enriched_rows)
print(f"\n✓ Enrichment complete: {len(result_df)} rows")
print(f"\nResults Summary:")
print(f"  Flagged (truck): {(result_df['truck_flag'] == True).sum()}")
print(f"  High confidence: {(result_df['confidence'] == 'high').sum()}")
print(f"  Errors: {(result_df['error'] != '').sum()}")

In [ ]:
# --- Load Data from Glue Catalog or Local CSV ---

# Option 1: Read from Glue Catalog (if running in Glue context)
# try:
#     spark = SparkSession.builder.appName("RegridEnrichment").getOrCreate()
#     glue_context = GlueContext(spark.sparkContext)
#     df_spark = glue_context.create_dynamic_frame.from_catalog(
#         database=INPUT_DATABASE, table_name=INPUT_TABLE
#     ).toDF()
#     df = df_spark.toPandas()
#     print(f"✓ Loaded from Glue: {len(df)} rows from {INPUT_DATABASE}.{INPUT_TABLE}")
# except Exception as e:
#     print(f"✗ Glue load failed: {e}")
#     df = None

# Option 2: Load from local CSV (for testing / non-Glue env)
# Create sample prospect data if running locally
sample_data = {
    "point_id": ["P001", "P002", "P003", "P004", "P005"],
    "lat": [41.5772, 34.0633, 39.7684, 41.7015, 36.3729],
    "lon": [-90.7477, -117.6509, -86.1581, -71.1550, -94.2088],
    "city": ["Walcott", "Ontario", "Indianapolis", "Providence", "Bentonville"],
    "state": ["IA", "CA", "IN", "RI", "AR"],
}

df = pd.DataFrame(sample_data)

if LIMIT_ROWS:
    df = df.head(LIMIT_ROWS)

print(f"✓ Loaded {len(df)} prospects")
print(f"\nColumns: {list(df.columns)}")
print(f"Sample:")
print(df.head(3))

In [ ]:
# --- Inline Regrid Client ---

class RegridClient:
    """Minimal Regrid API client with rate limiting and retries."""
    
    def __init__(self, query_url, min_interval=0.2, max_retries=3):
        self.query_url = query_url
        self.min_interval = min_interval
        self.max_retries = max_retries
        self._last_call = 0.0
        self.out_fields = [
            "address", "owner", "zoning_type", "zoning_subtype",
            "lbcs_activity", "lbcs_function", "county", "state2",
        ]
    
    def _throttle(self):
        """Rate limit: space requests min_interval apart."""
        if self.min_interval <= 0:
            return
        elapsed = time.monotonic() - self._last_call
        wait = self.min_interval - elapsed
        if wait > 0:
            time.sleep(wait)
        self._last_call = time.monotonic()
    
    def query_point(self, lat, lon):
        """Query Regrid for the parcel at (lat, lon). Returns attributes dict or None."""
        if requests is None:
            raise RuntimeError("requests library required")
        
        geom = json.dumps({"x": lon, "y": lat, "spatialReference": {"wkid": 4326}})
        params = {
            "f": "json",
            "geometry": geom,
            "geometryType": "esriGeometryPoint",
            "inSR": "4326",
            "spatialRel": "esriSpatialRelIntersects",
            "outFields": ",".join(self.out_fields),
            "returnGeometry": "false",
        }
        
        last_exc = None
        for attempt in range(self.max_retries):
            self._throttle()
            try:
                resp = requests.get(self.query_url, params=params, timeout=15)
                if resp.status_code == 429:
                    wait = (1.5 ** attempt) + random.uniform(0, 0.5)
                    print(f"  [rate limited] waiting {wait:.1f}s...")
                    time.sleep(wait)
                    continue
                resp.raise_for_status()
                data = resp.json()
                if "error" in data:
                    raise Exception(f"Regrid error: {data['error']}")
                feats = data.get("features", [])
                return feats[0].get("attributes", {}) if feats else None
            except Exception as exc:
                last_exc = exc
                if attempt < self.max_retries - 1:
                    wait = (1.5 ** attempt) + random.uniform(0, 0.5)
                    time.sleep(wait)
        
        raise Exception(f"Regrid failed after {self.max_retries} retries: {last_exc}")

# Initialize client
if not USE_MOCK:
    client = RegridClient(REGRID_QUERY_URL, min_interval=MIN_INTERVAL_SECS)
else:
    client = None

print("✓ Regrid client initialized")

In [ ]:
# --- Inline Classifier: Parcel truck/logistics detection ---

TRUCK_KEYWORDS = [
    r"\bTRUCK\s*STOP\b", r"\bTRUCKSTOP\b", r"\bTRUCKING\b", r"\bTRUCK\s*TERMINAL\b",
    r"\bFREIGHT\b", r"\bLOGISTICS\b", r"\bDISTRIBUTION\b", r"\bWAREHOUS",
    r"\bTRANSLOAD\b", r"\bINTERMODAL\b", r"\bTERMINAL\b",
    r"\bTRAVEL\s*CENTER\b", r"\bTRAVEL\s*PLAZA\b", r"\bTRAVEL\s*STOP\b",
    r"\bWEIGH\s*STATION\b", r"\bTRUCK\s*WASH\b",
    r"\bPILOT\b", r"\bFLYING\s*J\b", r"\bLOVE'?S\b", r"\bPETRO\b", r"\bTA\s*TRAVEL\b",
    r"\bIOWA\s*80\b", r"\bFEDEX\b", r"\bUPS\b",
]
TRUCK_KEYWORD_RE = re.compile("|".join(TRUCK_KEYWORDS), re.IGNORECASE)

WEAK_KEYWORDS = [r"\bFUEL\b", r"\bTRAVEL\b", r"\bPLAZA\b", r"\bGAS\b", r"\bSTATION\b"]
WEAK_KEYWORD_RE = re.compile("|".join(WEAK_KEYWORDS), re.IGNORECASE)

def classify_parcel(attrs):
    """
    Classify a parcel as truck/logistics-likely.
    
    attrs: dict from Regrid API (or empty dict)
    returns: {"truck_flag": bool, "confidence": str, "reasons": [str]}
    """
    attrs = attrs or {}
    reasons = []
    
    # Extract text fields
    text_fields = " ".join(str(attrs.get(f) or "") for f in ("owner", "address", "struct", "zoning_subtype"))
    
    # Tier 1: Strong keywords
    strong_kw = TRUCK_KEYWORD_RE.search(text_fields)
    if strong_kw:
        reasons.append(f"keyword: '{strong_kw.group(0)}'")
    
    # Tier 2: Industrial zoning
    zoning_type = (attrs.get("zoning_type") or "").strip().lower()
    is_industrial = zoning_type in {"industrial", "manufacturing", "warehouse"}
    if is_industrial:
        reasons.append(f"zoning: {attrs.get('zoning_type')}")
    
    # Tier 3: LBCS codes (3000-3999 mfg/warehouse, 4000-4999 transport)
    lbcs_hit = False
    for field in ("lbcs_activity", "lbcs_function"):
        code = attrs.get(field)
        if code:
            try:
                n = int(str(code)[:4])
                if 3000 <= n <= 4999:
                    reasons.append(f"{field}={code}")
                    lbcs_hit = True
                    break
            except (ValueError, TypeError):
                pass
    
    # Tier 4: Special commercial + weak keywords
    weak_kw = WEAK_KEYWORD_RE.search(text_fields)
    zoning_subtype = (attrs.get("zoning_subtype") or "").strip().lower()
    special_commercial = "special commercial" in zoning_subtype
    
    # Decision logic
    if strong_kw:
        conf = "high" if (is_industrial or lbcs_hit) else "medium"
        return {"truck_flag": True, "confidence": conf, "reasons": reasons}
    
    if is_industrial or lbcs_hit:
        conf = "high" if (is_industrial and lbcs_hit) else "medium"
        return {"truck_flag": True, "confidence": conf, "reasons": reasons}
    
    if special_commercial and weak_kw:
        reasons.append(f"special commercial + weak keyword: '{weak_kw.group(0)}'")
        return {"truck_flag": True, "confidence": "low", "reasons": reasons}
    
    return {"truck_flag": False, "confidence": "high", "reasons": ["no truck signal"]}

print("✓ Classifier loaded")

In [ ]:
# --- Configuration ---

# Regrid API
REGRID_QUERY_URL = os.environ.get("REGRID_QUERY_URL")
USE_MOCK = not REGRID_QUERY_URL  # Fall back to mock if URL not set

# Rate limiting
MIN_INTERVAL_SECS = 0.2  # Adjust based on your Regrid plan
MAX_RETRIES = 3

# Input/Output
INPUT_DATABASE = "default"  # Glue Catalog database
INPUT_TABLE = "prospects"    # Glue Catalog table (columns: point_id, lat, lon, ...)
OUTPUT_S3_PATH = "s3://your-bucket/regrid-enriched/"  # S3 output path
OUTPUT_LOCAL_CSV = "/tmp/enriched_prospects.csv"  # Local staging

# Limits
LIMIT_ROWS = None  # Set to 100 for testing; None for all

print("[config] Mode: " + ("MOCK" if USE_MOCK else "LIVE REGRID API"))
print(f"[config] Input: {INPUT_DATABASE}.{INPUT_TABLE}")
print(f"[config] Output: {OUTPUT_S3_PATH}")
print(f"[config] Rate limit: {1/MIN_INTERVAL_SECS:.2f} req/sec")
if LIMIT_ROWS:
    print(f"[config] Limit: {LIMIT_ROWS} rows")

In [ ]:
import sys
import os
import json
import re
import time
import random
from datetime import datetime

# AWS Glue imports
try:
    import awswrangler as wr
    from awsglue.context import GlueContext
    from awsglue.job import Job
    from pyspark.sql import SparkSession
except ImportError:
    print("⚠️  AWS Glue libraries not available (running locally)")
    wr = None
    GlueContext = None

# Data processing
import pandas as pd

# API client
try:
    import requests
except ImportError:
    requests = None
    print("⚠️  requests library not installed")

print("✓ Imports successful")

# Regrid Parcel Enrichment - Glue Notebook (Minimal)

Self-contained AWS Glue notebook for enriching prospect locations with Regrid parcel data.

**No external dependencies** — all logic is inline, works directly with Glue Catalog tables.

## Workflow

1. Read prospect table from Glue Catalog (with `lat`, `lon` columns)
2. Query Regrid API for parcel data at each point
3. Classify parcel as truck/logistics-likely (based on zoning, keywords, LBCS codes)
4. Write enriched CSV to S3

## Setup

1. Set Regrid API URL:
   ```bash
   export REGRID_QUERY_URL="https://fs.regrid.com/<your-token>/rest/services/premium/FeatureServer/0/query"
   ```

2. Configure Glue job parameters (see cell 2)